# Import necessory Packages

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.svm import LinearSVC

from sklearn.metrics import classification_report, confusion_matrix

# Load the Dataset

In [2]:
path = r'/content/drive/MyDrive/Colab Notebooks/HDFS_2k.log_structured.csv'

try:
    print("Processing.....\n")
    df = pd.read_csv(path)
    print("Dataset Loaded successfully......\n")
except FileExistsError:
    print("Please fix the path or Dataset does not exist")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Processing.....

Dataset Loaded successfully......



# Feature Engineering

In [3]:
time_str = df['Time'].astype(str).str.zfill(6)
df['Hour'] = time_str.str[:2].astype(int)
df['Minute'] = time_str.str[2:4].astype(int)
df['Second'] = time_str.str[4:].astype(int)

# Calculate Timestamp in seconds for Time Gap
def parse_time(t):
    t_str = str(t).zfill(6.0) # simplified parsing
    return int(t_str[:2])*3600 + int(t_str[2:4])*60 + int(t_str[4:])


# Time gap between consecutive log entries
timestamp = df['Time'].apply(lambda x: int(str(x).zfill(6)[:2])*3600 + int(str(x).zfill(6)[2:4])*60 + int(str(x).zfill(6)[4:]))
df['Time_gap'] = timestamp.diff().fillna(0)

# PID frequency (PID-related statistics)
pid_counts = df['Pid'].value_counts().to_dict()
df['Pid_freq'] = df['Pid'].map(pid_counts)

In [4]:
text_col = 'Content'
cat_cols = ['Component']
num_cols = ['Hour', 'Minute', 'Second', 'Time_gap', 'Pid_freq']
target_col = 'Level'

X = df[[text_col] + cat_cols + num_cols]
y = df[target_col]

# Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=1500, stop_words='english'), text_col),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

# 5. Model Pipeline with Class Weighting for Imbalance
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LinearSVC(class_weight='balanced', random_state=42, max_iter=10000))
])

In [7]:
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

In [8]:
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

        INFO       1.00      1.00      1.00       384
        WARN       1.00      1.00      1.00        16

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400

Confusion Matrix:
[[384   0]
 [  0  16]]


# Save

In [9]:
import joblib

model_filename = 'hdfs_log_classifier.joblib'
joblib.dump(model_pipeline, model_filename)

print(f"Model successfully saved to {model_filename}")

Model successfully saved to hdfs_log_classifier.joblib


In [10]:
import joblib
import pandas as pd

# Load the saved model pipeline
loaded_model = joblib.load('hdfs_log_classifier.joblib')

# Example new log data matching your feature engineering structure
new_data = pd.DataFrame({
    'Content': ['PacketResponder 1 for block blk_38865049 terminating'],
    'Component': ['dfs.DataNode$PacketResponder'],
    'Hour': [20],
    'Minute': [36],
    'Second': [15],
    'Time_gap': [0.0],
    'Pid_freq': [1]
})

# Predict the level
prediction = loaded_model.predict(new_data)
print(f"Predicted Level: {prediction[0]}")

Predicted Level: INFO
